## Weekly shift handover (live MongoDB)

Loads shift reports from the CastNet **`reports`** collection for a date range, then summarises **only equipment flagged DCM** in the CastNet equipment list (`dcm: true`), in **numerical machine order** (1, 2, … 10, not lexicographic). DCM ladders and other assets that are not flagged are skipped. Configure MongoDB via env vars (`CAST_LLM_MONGODB_URI`, etc.) or defaults in [settings.py](../src/cast_llm/settings.py).

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

from IPython.display import Markdown, display

from cast_llm import fetch_reports_date_range, group_by_equipment
from cast_llm.services.weekly_handover import (
    generate_weekly_handover,
    load_weekly_system_prompt,
)

### Date range (UK plant; Mongo `date_iso` is UTC)

Set **inclusive** `start_local` and `end_local` in `Europe/London` (any span: e.g. Sun–Sat week, Sunday shift start to midweek, or **Wed 06:00 to next Wed 06:00**). The service matches the API: same window logic as `POST /handover/weekly` with `start_local` / `end_local`.

In [ ]:
UK = ZoneInfo("Europe/London")
UTC = timezone.utc

# UK week Sun 12 Apr – Sat 18 Apr 2026 (inclusive)
start_local = datetime(2026, 4, 19, 0, 0, 0, tzinfo=UK)
end_local = datetime(2026, 4, 25, 23, 59, 59, tzinfo=UK)

start_utc = start_local.astimezone(UTC)
end_utc = end_local.astimezone(UTC)
print("Query UTC range:", start_utc, "→", end_utc)

### Fetch from MongoDB and group by DCM

In [ ]:
from cast_llm.mongo_reports import fetch_dcm_equipment_names
from cast_llm.services.weekly_handover import select_dcm_handover_equipment

rows = fetch_reports_date_range(start_utc, end_utc)
by_dcm = group_by_equipment(rows)
dcm_equipment_order = select_dcm_handover_equipment(
    by_dcm,
    fetch_dcm_equipment_names(),
)

print(f"Total reports in range: {len(rows)}")
print(f"DCM-only equipment keys (numerical order): {len(dcm_equipment_order)}")
for k in dcm_equipment_order:
    print(f"  {k}: {len(by_dcm[k])} report(s)")

### System prompt

In [ ]:
system_prompt = load_weekly_system_prompt()
print(f"Loaded system prompt ({len(system_prompt)} chars)")

### Run the models (one call per DCM per model)

`HANDOVER_MODELS` lists Ollama tags to compare (e.g. `gemma4:latest`, `llama4:scout` from `ollama list`). Outputs are saved under **`/home/castalum/thinclient_drives/AI Serve/`** as separate markdown files.

**Plant glossary:** `generate_weekly_handover` automatically loads `artifacts/shift_glossary/finalglossary.json` (term / expansion / definition) into handover context—the same behaviour as CastNet’s `POST /handover/weekly`. Set `use_glossary=False` to disable, or pass `glossary_path=` to override.

In [ ]:
HANDOVER_MODELS = [
    "gemma4:latest",
]

OUTPUT_DIR = Path("/home/castalum/thinclient_drives/AI Serve")

handover_outputs: dict[str, str] = {}
handover_json_outputs: dict[str, dict] = {}

for ollama_model in HANDOVER_MODELS:
    print(f"=== Model: {ollama_model} ===")
    run = generate_weekly_handover(
        start_local=start_local,
        end_local=end_local,
        model=ollama_model,
        timezone_name="Europe/London",
    )
    handover_json_outputs[ollama_model] = run

    sections: list[str] = [
        f"# Weekly handover summary\n\n"
        f"**Model:** `{run['model']}`  \n"
        f"**UTC range:** {run['start_utc']} – {run['end_utc']}  \n"
        f"**Local window:** {run['start_local']} – {run['end_local']}\n"
    ]

    for machine in run["machines"]:
        label = machine["equipment"]
        if machine.get("error"):
            body = f"Generation error: {machine['error']}"
        else:
            body = machine["summary"]
        sections.append(f"## {label}\n\n{body}\n")
        print(f"  Done: {label} ({machine['report_count']} reports)")

    handover_outputs[ollama_model] = "\n---\n\n".join(sections)
    print(f"Finished: {ollama_model}\n")

### Preview

In [ ]:
for model_name, text in handover_outputs.items():
    display(Markdown(f"## Preview: `{model_name}`\n\n---\n\n{text}"))

### Save markdown (comparison files)

Writes one `.md` per model to **`/home/castalum/thinclient_drives/AI Serve/`** (creates the folder if needed). Filenames: `weekly_handover_<model_tag>.md`.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for model_name, text in handover_outputs.items():
    safe = model_name.replace(":", "_").replace("/", "_")
    out_path = OUTPUT_DIR / f"weekly_handover_{safe}.md"
    out_path.write_text(text, encoding="utf-8")
    print("Wrote", out_path)